# Understanding evolution in the $P-\dot{P}$

With this notebook we show how the different ingredients in the magneto-rotational evolution play a role in determining the evolutionary tracks in the $P-\dot{P}$ diagram.
In particular we show the evolutionary tracks for two model pulsars where the initial properties ($P_0$, $B_0$ and $\chi_0$) can be changed at will. We can see how the magnetic-field decay and the alignment shape the trajectories for different values of the initial properties.

Here we summarize the outcome for some meaningful example cases.
Choosing different magnetic fields for the two stars will make the stars evolve on very different tracks and is therefore the most relevant and evident effect. In the following we will keep the same magnetic field value for both stars and we will focus on more subtle effects like the alignment and the choice of initial spin period and inclination angle.

## turning off the alignment

Let's fix the inclination angle to $\pi/2$ for both stars and consider how choosing different initial spin periods affects the results depending on the initial magnetic field. For example consider the following spin periods: 

P_initial_test = np.array([0.01, 0.1])

and try different field values $10^{12}$, $10^{13}$, $10^{14}$ G (the same for both stars).

For lower fields the track stay separated as the spin-down is not very effective and the field decay. Therefore the initial spin period is still imprinted in the final outcome.
For higher fields the pulsars end up evolving on the same evolutionary track and end up with the same final spin period. The evolution loses completely memory of the initial spin period.

If we consider two extreme values of the inclination angle:

chi_initial_test = np.array([0 * np.pi / 2, np.pi / 2])

This time the pulsars will evolve on slightly separeted tracks and end up with different final periods. 
However especially for higher fields, these final period values will not depend on the initial spin period values. Again the evolution loses memory of the initial spin periods for higher fields.

## With alignment

If the two pulsars start with the same inclination angle but different initial periods, for low magnetic fields the two evolutionary tracks are separated again because the spin-down is not very effective and the magnetic field decays. For higher magnetic fields the two stars again end up evolving on the same track and with the same final period. 

If the two pulsars start with very different inclination angle (one almost aligned and one almost orthogonal) and different initial periods the evolution tracks can stay separated even for higher magnetic field values. 
For example with this initial conditions:

B_initial_test = np.array([1e14, 1e14])

chi_initial_test = np.array([0.1 * np.pi / 2, 0.9 * np.pi / 2])

P_initial_test = np.array([0.01, 0.1])

This is due to the fact that the second star that is almost orthogonal and starts with a higher spin period does not reach the alignment state.
However this is very case-dependent because by inverting the spin period values for the two stars the two evolutionary tracks becomes asintotically very similar again.

In general the smaller is the initial spin period the easier is to lose its memory during the evolution.
Pulsars that are born with already high values of the spin period will more likely follow separated evolutionary tracks even for higher magnetic fields so that it is easier to differentiate between pulsars born with different spin periods. NOte that this is the case if the magnetic field decays. If it stays constant also these cases are degenerate.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy.optimize import curve_fit
from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import pypopsyn.simulator.multiband_emission.emission_radio as er

import utilities.plot_settings

import pypopsyn.simulator.magneto_rotational_physics.magneto_rotational_evolution_fit as mre
import pypopsyn.simulator.magneto_rotational_physics.period_derivative as pdv

from scipy.integrate import odeint

In [ ]:
cfg["NS_number"] = 2
cfg["t_age_max"] = 1e7

B_initial_test = np.array([1e13, 1e13])
chi_initial_test = np.array([0.1 * np.pi / 2, 0.1 * np.pi / 2])
P_initial_test = np.array([0.2, 0.6])
t_age_test = np.array([cfg["t_age_max"], cfg["t_age_max"]])
a_late = cfg["a_late"]

In [ ]:
P_dot_initial_test = np.zeros(2)

P_dot_initial_test[0] = (
    pdv.period_derivative(
        B_initial_test[0], chi_initial_test[0], P_initial_test[0]
    )
    / const.YR_TO_S
)

P_dot_initial_test[1] = (
    pdv.period_derivative(
        B_initial_test[1], chi_initial_test[1], P_initial_test[1]
    )
    / const.YR_TO_S
)

In [ ]:
def magneto_rotational_evolution_tracked_single_object(
    B_initial: float, chi_initial: float, P_initial: float, t_age: float, a_late: float
):
    # Initialization of the time grid.
    time_grid = np.append(
        10 ** np.arange(0, np.log10(t_age), cfg["magrot_time_step_log10"],), t_age,
    )

    B_asymptotic = 10 ** np.random.normal(
        cfg["B_millisec_mean"], cfg["B_millisec_sigma"], 1
    )[0]
    
    # Initial conditions for the three parameters.
    y_initial = np.array([chi_initial, P_initial])

    evol_output = np.array(
            odeint(
                mre.combined_derivatives,
                y0=y_initial,
                t=time_grid,
                args=(B_initial, B_asymptotic, a_late),
                tfirst=True,
            )
        )
    
    B = mre.magnetic_field_evolution_fit_numpy(
            B_initial, time_grid, B_asymptotic, a_late
        )
    
    chi = evol_output[:, 0]
    P = evol_output[:, 1]
        
    return B, chi, P, time_grid

In [ ]:
B_A, chi_A, P_A, t = magneto_rotational_evolution_tracked_single_object(
    B_initial_test[0], chi_initial_test[0], P_initial_test[0], t_age_test[0], a_late
)


B_B, chi_B, P_B, t = magneto_rotational_evolution_tracked_single_object(
    B_initial_test[1], chi_initial_test[1], P_initial_test[1], t_age_test[1], a_late
)

In [ ]:
period_derivative_vect = np.vectorize(pdv.period_derivative)

P_dot_A = period_derivative_vect(B_A, chi_A, P_A) / const.YR_TO_S
P_dot_B = period_derivative_vect(B_B, chi_B, P_B) / const.YR_TO_S

In [ ]:
P_final_test = np.array([P_A[-1], P_B[-1]])
P_dot_final_test = np.array([P_dot_A[-1], P_dot_B[-1]])

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.loglog(P_A, P_dot_A, "-", lw=3, color="tab:orange")
ax.loglog(P_B, P_dot_B, "-", lw=3, color="tab:red")
ax.loglog(P_initial_test, P_dot_initial_test, "o", color="tab:blue", ms=8)
ax.loglog(P_final_test, P_dot_final_test, "o", color="tab:green", ms=8)

ax.set_xlim(5e-3, 1e2)
ax.set_ylim(1e-22, 1e-8)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.loglog(t, B_A, "-", color="tab:orange", lw=3, label=r"$10^{12}$ [G]")
ax.loglog(t, B_B, "-", color="tab:red", lw=3, label=r"$10^{14}$ [G]")

ax.set_xlim(1, 1e8)

plt.xlabel(r"$t$ [yr]")
plt.ylabel(r"$B$ [G]")
plt.legend(loc=3)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(t, chi_A, "-", color="tab:orange", lw=3, label=r"$10^{12}$ [G]")
ax.plot(t, chi_B, "-", color="tab:red", lw=3, label=r"$10^{14}$ [G]")

ax.set_xlim(1, 1e8)
ax.set_xscale("log")
plt.xlabel(r"$t$ [yr]")
plt.ylabel(r"$\chi$ [rad]")
plt.legend(loc=3)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(t, P_A, "-", color="tab:orange", lw=3, label=r"$10^{12}$ [G]")
ax.plot(t, P_B, "-", color="tab:red", lw=3, label=r"$10^{14}$ [G]")

ax.set_xlim(1., 1e8)
ax.set_ylim(5e-3, 1e2)
ax.set_xscale("log")
ax.set_yscale("log")

plt.xlabel(r"$t$ [yr]")
plt.ylabel(r"$P$ [s]")
plt.legend(loc=3)

plt.show()